# LG Aimers — TabM Train Feature Set Experiment

목적:
- 공식 `train.csv`만 사용
- 기존 TabM baseline 전처리/학습 조건 유지
- Current Season `Level + Reliability`를 현재 champion feature set으로 사용
- 기존 baseline feature block drop-ablation 실행

현재 seed=42 기준 기록:
- Baseline Brier: `0.2490969012`
- Current Season Level + Reliability: `0.248446` (현재 champion)



In [ ]:
# Colab setup
!pip -q install tabm

In [ ]:
import copy
import random
import numpy as np
import pandas as pd
import torch

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.metrics import brier_score_loss, roc_auc_score

from torch.utils.data import TensorDataset, DataLoader
from tabm import TabM

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("device:", device)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 1. Load official train.csv

In [ ]:
TRAIN_PATH = "/content/train.csv"
TARGET_COL = "control_success"

train = pd.read_csv(TRAIN_PATH)

print("train shape:", train.shape)
print("season range:", train["season"].min(), "~", train["season"].max())
print("2024 rows:", (train["season"] == 2024).sum())

## 2. Baseline clean features

In [ ]:
train_clean = train.copy()

# Missing-history flags
train_clean["pitcher_recent_history_missing"] = (
    train_clean["asof_pitcher_prev1_game_success_rate"].isna()
).astype("int8")

train_clean["pitcher_history_missing"] = (
    train_clean["asof_pitcher_success_rate"].isna()
).astype("int8")

train_clean["batter_history_missing"] = (
    train_clean["asof_batter_success_rate"].isna()
).astype("int8")

# Log features
train_clean["log1p_asof_pitcher_n"] = np.log1p(
    train_clean["asof_pitcher_n"]
)

train_clean["log1p_asof_batter_n"] = np.log1p(
    train_clean["asof_batter_n"]
)

train_clean["log1p_li"] = np.log1p(
    train_clean["li"]
)

# Original TabM baseline excluded pitchmix_n itself.
if "asof_pitcher_pitchmix_n" in train_clean.columns:
    train_clean = train_clean.drop(columns=["asof_pitcher_pitchmix_n"])

print("train_clean shape:", train_clean.shape)

In [ ]:
# Original baseline typing recovered from the existing notebook

BASELINE_CATEGORICAL_COLS = [
    "game_month",
    "game_dayofweek",
    "inning",
    "top_bottom",
    "game_type",
    "balls_before",
    "strikes_before",
    "outs_before",
    "base_state",
    "pitcher_hand",
    "batter_hand",
    "pitcher_team_id",
    "batter_team_id",
]

BASELINE_BINARY_COLS = [
    "runner_on_1b",
    "runner_on_2b",
    "runner_on_3b",
    "pitcher_recent_history_missing",
    "pitcher_history_missing",
    "batter_history_missing",
]

BASELINE_NUMERIC_COLS = [
    "run_top_before",
    "run_bot_before",
    "run_total_before",
    "score_diff_home",
    "score_diff_pitcher_team",
    "num_runners_on",
    "home_win_expectancy",
    "li",

    "asof_pitcher_n",
    "asof_pitcher_success_rate",
    "asof_pitcher_reverse_rate",
    "asof_pitcher_middle_rate",
    "asof_pitcher_ball_rate",
    "asof_pitcher_strike_rate",

    "asof_pitcher_prev1_game_success_rate",
    "asof_pitcher_prev3_game_success_rate",
    "asof_pitcher_prev5_game_success_rate",

    "asof_pitcher_prev1_game_middle_rate",
    "asof_pitcher_prev3_game_middle_rate",
    "asof_pitcher_prev5_game_middle_rate",

    "asof_batter_n",
    "asof_batter_success_rate",
    "asof_batter_middle_rate",

    "asof_pitcher_fastball_rate",
    "asof_pitcher_breaking_rate",
    "asof_pitcher_offspeed_rate",

    "log1p_asof_pitcher_n",
    "log1p_asof_batter_n",
    "log1p_li",
]

print("Baseline numeric     :", len(BASELINE_NUMERIC_COLS))
print("Baseline binary      :", len(BASELINE_BINARY_COLS))
print("Baseline categorical :", len(BASELINE_CATEGORICAL_COLS))
print("Baseline total       :", (
    len(BASELINE_NUMERIC_COLS)
    + len(BASELINE_BINARY_COLS)
    + len(BASELINE_CATEGORICAL_COLS)
))

## 3. Build `tabm_core`

In [ ]:
tabm_core = train_clean.copy()

# 1) Recent vs career delta
for w in [1, 3, 5]:
    tabm_core[f"pitcher_success_prev{w}_minus_career"] = (
        tabm_core[f"asof_pitcher_prev{w}_game_success_rate"]
        - tabm_core["asof_pitcher_success_rate"]
    )

for w in [1, 3, 5]:
    tabm_core[f"pitcher_middle_prev{w}_minus_career"] = (
        tabm_core[f"asof_pitcher_prev{w}_game_middle_rate"]
        - tabm_core["asof_pitcher_middle_rate"]
    )

# 2) Recover career success count
tabm_core["asof_pitcher_success_count"] = (
    tabm_core["asof_pitcher_n"]
    * tabm_core["asof_pitcher_success_rate"]
).round()

# 3) Season-start career cumulative values
season_start_table = (
    tabm_core
    .sort_values(["pitcher_id", "season", "asof_pitcher_n"])
    .groupby(["pitcher_id", "season"], as_index=False)
    .first()[
        [
            "pitcher_id",
            "season",
            "asof_pitcher_n",
            "asof_pitcher_success_count",
        ]
    ]
    .rename(
        columns={
            "asof_pitcher_n": "prior_season_end_pitcher_n",
            "asof_pitcher_success_count":
                "prior_season_end_pitcher_success_count",
        }
    )
)

tabm_core = tabm_core.merge(
    season_start_table,
    on=["pitcher_id", "season"],
    how="left",
    validate="many_to_one",
)

# 4) Current-season counts
tabm_core["pitcher_current_season_n"] = (
    tabm_core["asof_pitcher_n"]
    - tabm_core["prior_season_end_pitcher_n"]
)

tabm_core["pitcher_current_season_success_count"] = (
    tabm_core["asof_pitcher_success_count"]
    - tabm_core["prior_season_end_pitcher_success_count"]
)

# 5) Current-season raw rate
tabm_core["pitcher_current_season_success_rate"] = np.where(
    tabm_core["pitcher_current_season_n"] > 0,
    (
        tabm_core["pitcher_current_season_success_count"]
        / tabm_core["pitcher_current_season_n"]
    ),
    np.nan,
)

# 6) Reliability
tabm_core["log1p_pitcher_current_season_n"] = np.log1p(
    tabm_core["pitcher_current_season_n"]
)

tabm_core["pitcher_current_season_available_flag"] = (
    tabm_core["pitcher_current_season_n"] > 0
).astype("int8")

SMALL_SAMPLE_THRESHOLD = 30

tabm_core["pitcher_current_season_small_sample_flag"] = (
    tabm_core["pitcher_current_season_n"]
    .between(1, SMALL_SAMPLE_THRESHOLD - 1)
).astype("int8")

# 7) Smoothed current-season rate
SMOOTHING_STRENGTH = 50

tabm_core["pitcher_current_season_success_rate_smoothed"] = (
    tabm_core["pitcher_current_season_success_count"]
    + SMOOTHING_STRENGTH * tabm_core["asof_pitcher_success_rate"]
) / (
    tabm_core["pitcher_current_season_n"]
    + SMOOTHING_STRENGTH
)

# 8) Relative features
tabm_core["pitcher_current_season_success_minus_career"] = (
    tabm_core["pitcher_current_season_success_rate"]
    - tabm_core["asof_pitcher_success_rate"]
)

tabm_core["pitcher_current_season_success_minus_prev5"] = (
    tabm_core["pitcher_current_season_success_rate"]
    - tabm_core["asof_pitcher_prev5_game_success_rate"]
)

tabm_core["pitcher_current_season_n_ratio_to_career"] = (
    tabm_core["pitcher_current_season_n"]
    / (tabm_core["asof_pitcher_n"] + 1)
)

print("tabm_core shape:", tabm_core.shape)

## 4. Current Season blocks

In [ ]:
CURRENT_LEVEL = [
    "pitcher_current_season_success_rate",
    "pitcher_current_season_success_rate_smoothed",
]

CURRENT_RELIABILITY_NUMERIC = [
    "pitcher_current_season_n",
    "pitcher_current_season_success_count",
    "log1p_pitcher_current_season_n",
    "pitcher_current_season_n_ratio_to_career",
]

CURRENT_RELIABILITY_BINARY = [
    "pitcher_current_season_available_flag",
    "pitcher_current_season_small_sample_flag",
]

CURRENT_RELATIVE = [
    "pitcher_current_season_success_minus_career",
    "pitcher_current_season_success_minus_prev5",
]

# Current seed-42 champion = Baseline + Level + Reliability
CHAMPION_NUMERIC = (
    BASELINE_NUMERIC_COLS
    + CURRENT_LEVEL
    + CURRENT_RELIABILITY_NUMERIC
)

CHAMPION_BINARY = (
    BASELINE_BINARY_COLS
    + CURRENT_RELIABILITY_BINARY
)

CHAMPION_CATEGORICAL = BASELINE_CATEGORICAL_COLS.copy()

print("Champion numeric     :", len(CHAMPION_NUMERIC))
print("Champion binary      :", len(CHAMPION_BINARY))
print("Champion categorical :", len(CHAMPION_CATEGORICAL))
print("Champion total       :", (
    len(CHAMPION_NUMERIC)
    + len(CHAMPION_BINARY)
    + len(CHAMPION_CATEGORICAL)
))

## 5. Reusable preprocessing

In [ ]:
def prepare_tabm_data(
    df,
    numeric_cols,
    binary_cols,
    categorical_cols,
    target_col=TARGET_COL,
):
    train_df = df[df["season"] < 2024].copy()
    valid_df = df[df["season"] == 2024].copy()

    # Numeric: fit only on training seasons
    numeric_imputer = SimpleImputer(strategy="median")

    X_train_num = numeric_imputer.fit_transform(
        train_df[numeric_cols]
    )
    X_valid_num = numeric_imputer.transform(
        valid_df[numeric_cols]
    )

    numeric_scaler = StandardScaler()

    X_train_num = numeric_scaler.fit_transform(
        X_train_num
    ).astype(np.float32)

    X_valid_num = numeric_scaler.transform(
        X_valid_num
    ).astype(np.float32)

    # Binary
    X_train_bin = (
        train_df[binary_cols].to_numpy(dtype=np.float32)
    )
    X_valid_bin = (
        valid_df[binary_cols].to_numpy(dtype=np.float32)
    )

    X_train_cont = np.hstack(
        [X_train_num, X_train_bin]
    ).astype(np.float32)

    X_valid_cont = np.hstack(
        [X_valid_num, X_valid_bin]
    ).astype(np.float32)

    # Categorical
    train_cat = train_df[categorical_cols].astype(str)
    valid_cat = valid_df[categorical_cols].astype(str)

    categorical_encoder = OrdinalEncoder(
        handle_unknown="use_encoded_value",
        unknown_value=-1,
    )

    X_train_cat = (
        categorical_encoder
        .fit_transform(train_cat)
        .astype(np.int64)
        + 1
    )

    X_valid_cat = (
        categorical_encoder
        .transform(valid_cat)
        .astype(np.int64)
        + 1
    )

    cat_cardinalities = [
        len(categories) + 1
        for categories in categorical_encoder.categories_
    ]

    y_train = train_df[target_col].to_numpy(dtype=np.float32)
    y_valid = valid_df[target_col].to_numpy(dtype=np.float32)

    return {
        "X_train_cont": X_train_cont,
        "X_valid_cont": X_valid_cont,
        "X_train_cat": X_train_cat,
        "X_valid_cat": X_valid_cat,
        "y_train": y_train,
        "y_valid": y_valid,
        "cat_cardinalities": cat_cardinalities,
    }

## 6. Reusable TabM training

In [ ]:
def train_tabm(
    data,
    seed=42,
    epochs=10,
    patience=3,
    batch_size=4096,
):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    X_train_num_t = torch.tensor(
        data["X_train_cont"], dtype=torch.float32
    )
    X_valid_num_t = torch.tensor(
        data["X_valid_cont"], dtype=torch.float32
    )

    X_train_cat_t = torch.tensor(
        data["X_train_cat"], dtype=torch.long
    )
    X_valid_cat_t = torch.tensor(
        data["X_valid_cat"], dtype=torch.long
    )

    y_train_t = torch.tensor(
        data["y_train"], dtype=torch.float32
    )
    y_valid_t = torch.tensor(
        data["y_valid"], dtype=torch.float32
    )

    train_dataset = TensorDataset(
        X_train_num_t, X_train_cat_t, y_train_t
    )
    valid_dataset = TensorDataset(
        X_valid_num_t, X_valid_cat_t, y_valid_t
    )

    g = torch.Generator()
    g.manual_seed(seed)

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=0,
        pin_memory=torch.cuda.is_available(),
        generator=g,
    )

    valid_loader = DataLoader(
        valid_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
        pin_memory=torch.cuda.is_available(),
    )

    model = TabM.make(
        n_num_features=data["X_train_cont"].shape[1],
        cat_cardinalities=data["cat_cardinalities"],
        d_out=1,
        n_blocks=3,
        d_block=256,
        dropout=0.1,
        k=32,
        arch_type="tabm",
    ).to(device)

    criterion = torch.nn.BCEWithLogitsLoss()
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=2e-3,
        weight_decay=1e-5,
    )

    best_brier = float("inf")
    best_state = None
    best_epoch = None
    patience_count = 0
    history = []

    for epoch in range(1, epochs + 1):
        model.train()
        train_loss_sum = 0.0
        train_count = 0

        for x_num, x_cat, y in train_loader:
            x_num = x_num.to(device, non_blocking=True)
            x_cat = x_cat.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            logits = model(x_num=x_num, x_cat=x_cat)

            if logits.ndim == 3 and logits.shape[-1] == 1:
                logits = logits.squeeze(-1)

            target = y.unsqueeze(1).expand_as(logits)
            loss = criterion(logits, target)

            loss.backward()
            optimizer.step()

            bs = y.size(0)
            train_loss_sum += loss.item() * bs
            train_count += bs

        train_loss = train_loss_sum / train_count

        model.eval()
        valid_pred_parts = []

        with torch.no_grad():
            for x_num, x_cat, _ in valid_loader:
                x_num = x_num.to(device, non_blocking=True)
                x_cat = x_cat.to(device, non_blocking=True)

                logits = model(x_num=x_num, x_cat=x_cat)

                if logits.ndim == 3 and logits.shape[-1] == 1:
                    logits = logits.squeeze(-1)

                probs = torch.sigmoid(logits).mean(dim=1)
                valid_pred_parts.append(probs.cpu().numpy())

        valid_pred = np.concatenate(valid_pred_parts)

        brier = brier_score_loss(
            data["y_valid"], valid_pred
        )
        auc = roc_auc_score(
            data["y_valid"], valid_pred
        )

        history.append({
            "epoch": epoch,
            "train_loss": train_loss,
            "brier": brier,
            "auc": auc,
            "pred_mean": float(valid_pred.mean()),
        })

        print(
            f"Epoch {epoch:02d} | "
            f"Brier {brier:.6f} | "
            f"AUC {auc:.6f} | "
            f"PredMean {valid_pred.mean():.6f}"
        )

        if brier < best_brier:
            best_brier = brier
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())
            patience_count = 0
        else:
            patience_count += 1

            if patience_count >= patience:
                print("Early stopping")
                break

    model.load_state_dict(best_state)

    return {
        "best_brier": float(best_brier),
        "best_epoch": int(best_epoch),
        "history": pd.DataFrame(history),
    }

## 7. DROP ablation — seed 42

현재 champion (`Baseline + Current Level + Reliability`)은 이미 검증했으므로 다시 학습하지 않고,
baseline 내부에서 상대적으로 제거 가능성이 있는 네 블록만 확인합니다.

In [ ]:
SEED = 42
CHAMPION_BRIER = 0.248446

DROP_RECENT_MIDDLE = [
    "asof_pitcher_prev1_game_middle_rate",
    "asof_pitcher_prev3_game_middle_rate",
    "asof_pitcher_prev5_game_middle_rate",
]

DROP_PITCH_MIX = [
    "asof_pitcher_fastball_rate",
    "asof_pitcher_breaking_rate",
    "asof_pitcher_offspeed_rate",
]

DROP_BATTER_HISTORY = [
    "asof_batter_n",
    "asof_batter_success_rate",
    "asof_batter_middle_rate",
    "log1p_asof_batter_n",
    "batter_history_missing",
]

DROP_WIN_EXPECTANCY_LI = [
    "home_win_expectancy",
    "li",
    "log1p_li",
]

DROP_EXPERIMENTS = {
    "Drop Recent Middle": DROP_RECENT_MIDDLE,
    "Drop Pitch Mix": DROP_PITCH_MIX,
    "Drop Batter History": DROP_BATTER_HISTORY,
    "Drop WinExp + LI": DROP_WIN_EXPECTANCY_LI,
}

drop_results = []

for name, drop_cols in DROP_EXPERIMENTS.items():
    print("\n" + "=" * 70)
    print(name)
    print("=" * 70)

    exp_numeric = [
        c for c in CHAMPION_NUMERIC
        if c not in drop_cols
    ]
    exp_binary = [
        c for c in CHAMPION_BINARY
        if c not in drop_cols
    ]
    exp_categorical = [
        c for c in CHAMPION_CATEGORICAL
        if c not in drop_cols
    ]

    data = prepare_tabm_data(
        tabm_core,
        numeric_cols=exp_numeric,
        binary_cols=exp_binary,
        categorical_cols=exp_categorical,
    )

    result = train_tabm(
        data,
        seed=SEED,
        epochs=10,
        patience=3,
        batch_size=4096,
    )

    brier = result["best_brier"]

    drop_results.append({
        "Experiment": name,
        "Dropped": len(drop_cols),
        "Brier": brier,
        "Delta_vs_Champion": brier - CHAMPION_BRIER,
        "Epoch": result["best_epoch"],
    })

drop_df = (
    pd.DataFrame(drop_results)
    .sort_values("Brier")
    .reset_index(drop=True)
)

print("\n" + "=" * 80)
print("TABM DROP ABLATION — SEED 42")
print("=" * 80)

print(
    f"{'Champion':<25}"
    f"Brier={CHAMPION_BRIER:.6f} | "
    f"Delta=+0.000000"
)

for _, row in drop_df.iterrows():
    print(
        f"{row['Experiment']:<25}"
        f"Brier={row['Brier']:.6f} | "
        f"Delta={row['Delta_vs_Champion']:+.6f} | "
        f"Dropped={int(row['Dropped'])} | "
        f"Epoch={int(row['Epoch'])}"
    )

drop_df

## Notes

- `season`, `pitcher_id`, `batter_id`는 기존 baseline model input에서 제외되어 있습니다.
- validation은 기존 실험과 동일하게 `season == 2024`입니다.
- numeric imputer/scaler 및 categorical encoder는 `season < 2024` 데이터에만 fit합니다.
- Current Season Relative 2개는 어제 ablation에서 champion에 추가했을 때 성능이 악화되어 현재 champion에서 제외했습니다.